# 🚀 Chaosops RL Training - Google Colab Complete Guide

**End-to-end training pipeline from GitHub to evaluation**

- ✅ GPU setup
- ✅ Clone repository
- ✅ Install dependencies
- ✅ Train RL agent
- ✅ Evaluate performance
- ✅ Visualize results

---

## 1️⃣ Check GPU & Environment

In [ ]:
import torch
import subprocess
import sys

print("🔍 Environment Check:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"  CUDA Version: {torch.version.cuda}")
else:
    print("  ⚠️  WARNING: No GPU detected. Enable GPU:")
    print("     Runtime → Change runtime type → GPU (T4 or V100)")

# Check free disk space
result = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True)
print(f"\n💾 Disk Space:")
lines = result.stdout.strip().split('\n')
if len(lines) > 1:
    print(f"  {lines[1]}")

## 2️⃣ Clone Repository from GitHub

In [ ]:
import os

# Clone repository
print("📥 Cloning Chaosops repository...")
result = subprocess.run(
    ["git", "clone", "https://github.com/orpheusdark/Chaosops.git", "/content/Chaosops"],
    capture_output=True,
    text=True,
    timeout=60
)

if result.returncode == 0:
    print("✅ Repository cloned successfully")
else:
    if "already exists" in result.stderr:
        print("✅ Repository already exists")
    else:
        print(f"❌ Clone failed: {result.stderr}")

# Verify files
os.chdir("/content/Chaosops")
print(f"\n📂 Repo contents:")
for item in os.listdir("."):
    if not item.startswith("."):
        print(f"   {item}/" if os.path.isdir(item) else f"   {item}")

print(f"\n📋 Python files in chaosops/:")
for f in sorted(os.listdir("chaosops")):
    if f.endswith(".py"):
        print(f"   ✓ {f}")

## 3️⃣ Install Dependencies (5-10 minutes)

In [ ]:
print("📦 Installing dependencies...\n")
print("This may take 5-10 minutes. Key packages:")
print("  • unsloth - Fast LLM finetuning")
print("  • transformers - Qwen2.5 model")
print("  • peft - LoRA adapters")
print("  • bitsandbytes - 4-bit quantization\n")

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "unsloth", "trl", "transformers", "datasets", "accelerate", "bitsandbytes", "peft", "torch"],
    capture_output=False,
    text=True,
    timeout=600
)

print("\n✅ Installation complete!")

## 4️⃣ Verify Installation

In [ ]:
print("🔧 Verifying imports...\n")

try:
    from unsloth import FastLanguageModel
    print("✓ unsloth")
except Exception as e:
    print(f"✗ unsloth: {e}")

try:
    from peft import get_peft_model, LoraConfig
    print("✓ peft (LoRA)")
except Exception as e:
    print(f"✗ peft: {e}")

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    print("✓ transformers (Qwen)")
except Exception as e:
    print(f"✗ transformers: {e}")

try:
    import torch
    print(f"✓ torch ({torch.__version__})")
except Exception as e:
    print(f"✗ torch: {e}")

print("\n✅ All packages ready for training!")

## 5️⃣ Run Training (10-15 minutes)

**What happens:**
1. Loads Qwen2.5-0.5B in 4-bit quantization with LoRA adapters
2. Initializes ChaosOps environment
3. Runs 10 grouped training episodes (GRPO-style)
4. Saves LoRA adapter to `chaosops-qwen-grpo/`

**Expected output:** Training progress logs with episode completion

In [ ]:
import os

os.chdir("/content/Chaosops/chaosops")

print("🚀 Starting Training...")
print("=" * 60)
print(f"Working directory: {os.getcwd()}")
print(f"Files present: {[f for f in os.listdir('.') if f.endswith('.py')]}")
print("=" * 60)
print()

result = subprocess.run(
    [sys.executable, "train.py", 
     "--episodes", "10",
     "--model_name", "Qwen/Qwen2.5-0.5B",
     "--output_dir", "../chaosops-qwen-grpo"],
    capture_output=False,
    text=True,
    timeout=1800  # 30 minute timeout
)

if result.returncode == 0:
    print("\n✅ Training completed successfully!")
else:
    print(f"\n❌ Training failed with return code {result.returncode}")

print("\nChecking saved adapter...")
adapter_path = "/content/Chaosops/chaosops-qwen-grpo"
if os.path.exists(adapter_path):
    files = os.listdir(adapter_path)
    print(f"✓ Adapter saved: {len(files)} files")
    for f in sorted(files)[:5]:
        print(f"  - {f}")
else:
    print(f"⚠️  Adapter directory not found at {adapter_path}")

## 6️⃣ Evaluate Model Performance (5-10 minutes)

**Evaluation tests:**
- **Baseline:** Random policy (5 random actions per episode)
- **Trained:** Loaded model with LoRA adapters
- **Variation:** Distribution shift test (schema drift, permission changes)

**Success metrics:**
- Success Rate (%)
- Average Reward
- Average Steps per Episode
- Error Type Counts

In [ ]:
import json

print("🧪 Running Evaluation...")
print("=" * 60)
print("Testing: Baseline vs Trained vs Variation")
print("Episodes per test: 20")
print("=" * 60)
print()

result = subprocess.run(
    [sys.executable, "eval.py",
     "--episodes", "20",
     "--adapter_dir", "../chaosops-qwen-grpo"],
    cwd="/content/Chaosops/chaosops",
    capture_output=True,
    text=True,
    timeout=900  # 15 minute timeout
)

if result.returncode == 0:
    try:
        # Extract JSON from output
        output_lines = result.stdout.strip().split('\n')
        json_str = '\n'.join(output_lines)
        eval_results = json.loads(json_str)
        
        print("✅ Evaluation completed successfully!\n")
    except json.JSONDecodeError as e:
        print(f"⚠️  Could not parse JSON: {e}")
        print(f"Raw output:\n{result.stdout}")
        eval_results = None
else:
    print(f"❌ Evaluation failed: {result.stderr}")
    eval_results = None

## 7️⃣ Display Results & Analysis

In [ ]:
if eval_results:
    print("\n" + "=" * 70)
    print("📊 CHAOSOPS RL TRAINING RESULTS".center(70))
    print("=" * 70)
    
    # Baseline
    print(f"\n🎲 BASELINE (Random Policy):")
    print(f"   Success Rate:      {eval_results['baseline']['success_rate']:>6.1%}")
    print(f"   Avg Reward:        {eval_results['baseline']['avg_reward']:>6.3f}")
    print(f"   Avg Steps:         {eval_results['baseline']['avg_steps']:>6.1f}")
    print(f"   Error Counts:")
    for err_type, count in eval_results['baseline']['error_counts'].items():
        print(f"      {err_type}: {count}")
    
    # Trained Model
    print(f"\n🤖 TRAINED MODEL (After RL Training):")
    print(f"   Success Rate:      {eval_results['trained']['success_rate']:>6.1%}")
    print(f"   Avg Reward:        {eval_results['trained']['avg_reward']:>6.3f}")
    print(f"   Avg Steps:         {eval_results['trained']['avg_steps']:>6.1f}")
    print(f"   Error Counts:")
    for err_type, count in eval_results['trained']['error_counts'].items():
        print(f"      {err_type}: {count}")
    
    # Variation Test (Robustness)
    print(f"\n🔄 VARIATION TEST (Distribution Shift):")
    print(f"   Success Rate:      {eval_results['variation']['success_rate']:>6.1%}")
    print(f"   Avg Reward:        {eval_results['variation']['avg_reward']:>6.3f}")
    print(f"   Avg Steps:         {eval_results['variation']['avg_steps']:>6.1f}")
    print(f"   Error Counts:")
    for err_type, count in eval_results['variation']['error_counts'].items():
        print(f"      {err_type}: {count}")
    
    # Improvements
    print(f"\n📈 IMPROVEMENTS (Trained vs Baseline):")
    print(f"   Success Improvement: +{eval_results['success_improvement']:>5.1%}")
    print(f"   Reward Improvement:  +{eval_results['reward_improvement']:>5.3f}")
    print(f"   Efficiency Gain:     -{eval_results['efficiency_gain']:>5.1f} steps")
    print(f"   Robustness Drop:     -{eval_results['robustness_drop']:>5.1%}")
    
    # Final Verdict
    print(f"\n" + "=" * 70)
    verdict = eval_results['verdict']
    if verdict == "ROBUST LEARNING":
        print(f"✅ VERDICT: {verdict} - Agent demonstrates true adaptive behavior!".center(70))
    elif verdict == "SCRIPTED POLICY":
        print(f"⚠️  VERDICT: {verdict} - Agent may have memorized policy.".center(70))
    else:
        print(f"❓ VERDICT: {verdict} - Agent shows limited learning.".center(70))
    print("=" * 70 + "\n")
else:
    print("\n❌ No evaluation results to display")

## 8️⃣ Visualization Dashboard

In [ ]:
if eval_results:
    import matplotlib.pyplot as plt
    import numpy as np
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('🎯 Chaosops RL Training Results', fontsize=18, fontweight='bold', y=0.995)
    
    # Colors
    colors = ['#FF6B6B', '#51CF66', '#4DABF7']  # Red (baseline), Green (trained), Blue (variation)
    policies = ['Baseline\n(Random)', 'Trained\nModel', 'Variation\nTest']
    
    # 1. Success Rate
    success_rates = [
        eval_results['baseline']['success_rate'],
        eval_results['trained']['success_rate'],
        eval_results['variation']['success_rate']
    ]
    ax = axes[0, 0]
    bars = ax.bar(policies, success_rates, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    ax.set_ylabel('Success Rate', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.15])
    ax.set_title('Success Rate Comparison', fontsize=13, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for i, (bar, v) in enumerate(zip(bars, success_rates)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.03,
                f'{v:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    # 2. Average Reward
    rewards = [
        eval_results['baseline']['avg_reward'],
        eval_results['trained']['avg_reward'],
        eval_results['variation']['avg_reward']
    ]
    ax = axes[0, 1]
    bars = ax.bar(policies, rewards, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    ax.set_ylabel('Average Reward', fontsize=12, fontweight='bold')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5, alpha=0.5)
    ax.set_title('Average Reward', fontsize=13, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for i, (bar, v) in enumerate(zip(bars, rewards)):
        height = bar.get_height()
        offset = 0.05 if v >= 0 else -0.08
        ax.text(bar.get_x() + bar.get_width()/2., height + offset,
                f'{v:.3f}', ha='center', va='bottom' if v >= 0 else 'top',
                fontweight='bold', fontsize=11)
    
    # 3. Episode Efficiency (Steps)
    steps = [
        eval_results['baseline']['avg_steps'],
        eval_results['trained']['avg_steps'],
        eval_results['variation']['avg_steps']
    ]
    ax = axes[1, 0]
    bars = ax.bar(policies, steps, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    ax.set_ylabel('Average Steps per Episode', fontsize=12, fontweight='bold')
    ax.set_title('Episode Efficiency (Lower is Better)', fontsize=13, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for i, (bar, v) in enumerate(zip(bars, steps)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.15,
                f'{v:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    # 4. Key Improvements
    ax = axes[1, 1]
    improvements_labels = ['Success\nImprovement', 'Reward\nImprovement', 'Efficiency\nGain\n(steps saved)', 'Robustness\nDrop']
    improvements_values = [
        eval_results['success_improvement'] * 100,  # as percentage
        eval_results['reward_improvement'],
        eval_results['efficiency_gain'],
        -eval_results['robustness_drop'] * 100  # negative for robustness drop
    ]
    improvements_colors = ['#51CF66' if v > 0 else '#FF6B6B' for v in improvements_values]
    bars = ax.bar(improvements_labels, improvements_values, color=improvements_colors, alpha=0.8, edgecolor='black', linewidth=2)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
    ax.set_ylabel('Magnitude', fontsize=12, fontweight='bold')
    ax.set_title('Training Improvements Summary', fontsize=13, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for bar, v in zip(bars, improvements_values):
        height = bar.get_height()
        offset = 3 if v >= 0 else -6
        label = f'{v:.1f}%' if 'Improvement' in bar.get_label() or 'Drop' in bar.get_label() else f'{v:.2f}'
        ax.text(bar.get_x() + bar.get_width()/2., height + offset,
                label, ha='center', va='bottom' if v >= 0 else 'top',
                fontweight='bold', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('/content/Chaosops/training_results_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ Dashboard saved to: /content/Chaosops/training_results_dashboard.png")
else:
    print("⚠️  Evaluation results not available for visualization")

## 9️⃣ Save Results to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil

try:
    print("📤 Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
    
    # Create results directory
    results_dir = "/content/drive/MyDrive/Chaosops_Results"
    os.makedirs(results_dir, exist_ok=True)
    print(f"✅ Results directory: {results_dir}\n")
    
    # Copy adapter
    print("📋 Copying trained adapter...")
    adapter_src = "/content/Chaosops/chaosops-qwen-grpo"
    adapter_dst = f"{results_dir}/chaosops-qwen-grpo"
    if os.path.exists(adapter_src):
        shutil.copytree(adapter_src, adapter_dst, dirs_exist_ok=True)
        print(f"✓ Adapter copied: {adapter_dst}")
    
    # Save evaluation results
    if eval_results:
        print("📋 Saving evaluation results...")
        with open(f"{results_dir}/eval_results.json", "w") as f:
            json.dump(eval_results, f, indent=2)
        print(f"✓ Eval results saved: eval_results.json")
    
    # Copy dashboard image
    dashboard_src = "/content/Chaosops/training_results_dashboard.png"
    if os.path.exists(dashboard_src):
        print("📋 Copying visualization dashboard...")
        shutil.copy(dashboard_src, f"{results_dir}/training_results_dashboard.png")
        print(f"✓ Dashboard copied: training_results_dashboard.png")
    
    print(f"\n✅ All results saved to Google Drive!")
    print(f"   Location: My Drive → Chaosops_Results")
    
except Exception as e:
    print(f"⚠️  Could not save to Google Drive: {e}")
    print("   (This is optional - results are still available in /content/Chaosops)")

## 🎯 Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("✅ TRAINING PIPELINE COMPLETE!".center(70))
print("="*70)

print("\n📊 What you've done:")
print("   1. ✓ Set up GPU environment")
print("   2. ✓ Cloned Chaosops repository from GitHub")
print("   3. ✓ Installed all dependencies (unsloth, peft, transformers, etc.)")
print("   4. ✓ Trained Qwen2.5-0.5B model with LoRA adapters")
print("   5. ✓ Evaluated: Baseline vs Trained vs Variation tests")
print("   6. ✓ Generated visualization dashboard")

if eval_results:
    print(f"\n📈 Your Results:")
    print(f"   Success Rate: {eval_results['baseline']['success_rate']:.1%} → {eval_results['trained']['success_rate']:.1%} (+{eval_results['success_improvement']:.1%})")
    print(f"   Avg Reward:   {eval_results['baseline']['avg_reward']:.3f} → {eval_results['trained']['avg_reward']:.3f} (+{eval_results['reward_improvement']:.3f})")
    print(f"   Efficiency:   {eval_results['baseline']['avg_steps']:.1f} → {eval_results['trained']['avg_steps']:.1f} steps (−{eval_results['efficiency_gain']:.1f})")
    print(f"   Verdict:      {eval_results['verdict']}")

print("\n🚀 Next Steps:")
print("   • To train longer: Change --episodes 10 to --episodes 50 (or more)")
print("   • To adjust learning rate: Edit train.py learning_rate parameter")
print("   • To run locally: Same code works on Windows/Mac with GPU")
print("   • Results auto-sync to GitHub: https://github.com/orpheusdark/Chaosops")
print("   • HF Space link: https://huggingface.co/spaces/orpheusdark/chaosops")

print("\n💾 Output files:")
print("   • /content/Chaosops/chaosops-qwen-grpo/ — Trained LoRA adapter")
print("   • /content/Chaosops/training_results_dashboard.png — Visualization")
print("   • Google Drive/Chaosops_Results/ — Backup copy (if saved)")

print("\n" + "="*70 + "\n")